# Verificación de la capa Silver — Iceberg

Revisamos la tabla `nessie.nyc.yellow_taxi` tras la limpieza:
referencias Nessie, metadatos Iceberg, snapshots, particionado y calidad de datos.
También comparamos contra bronze para validar las reglas de limpieza.

In [36]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie
from pyspark.sql.functions import (
    col, count, year as spark_year, month as spark_month,
    sum as spark_sum, round as spark_round, when,
)

spark = get_spark("verify-silver-iceberg")
spark.version

'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

In [37]:
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags     = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

Nessie references:

  * Branch bronze                                   8097a4dacf70
  * Branch main                                     69b0c632414e
  * Branch silver                                   f5e12552937a
    Tag    bronze-taxi_zone_lookup-2026             8097a4dacf70
    Tag    bronze-yellow_taxi-2019                  80d9ac5e044b
    Tag    bronze-yellow_taxi-2020                  b66a4f1f6855
    Tag    bronze-yellow_taxi-2022                  8c5b8217b632
    Tag    bronze-yellow_taxi-2023                  9a8380e67e3d
    Tag    bronze-yellow_taxi-2025                  7962320e85ff
    Tag    review/silver-yellow_taxi-2025-02        20100c27ba04
    Tag    review/silver-yellow_taxi-2025-03        00ced379718f
    Tag    silver-taxi_zone_lookup-2026             f5e12552937a
    Tag    silver-yellow_taxi-2019                  e1c02cf76f05
    Tag    silver-yellow_taxi-2020                  d801b54d9e40
    Tag    silver-yellow_taxi-2022                  e9d044586eaa
    T

In [38]:
# Switch to silver branch for the rest of the notebook
nessie.set_spark_branch(spark, "silver")

spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)
spark.sql("SHOW TABLES IN nessie.nyc").show(truncate=False)

+---------+
|namespace|
+---------+
|nyc      |
+---------+

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|nyc      |taxi_zone_lookup|false      |
|nyc      |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [39]:
spark.sql("DESCRIBE EXTENDED nessie.nyc.yellow_taxi").show(250, truncate=False)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                                                     |comment|
+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [40]:
spark.sql("SHOW TBLPROPERTIES nessie.nyc.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |5607906580105928636                                             |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |f5e12552937a7cc123ecb0f35c4e25369ae934ee3736c1d9ff5218d0884d4fb4|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [41]:
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                      

In [42]:
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-17 16:20:38.469|7137748156835262556|NULL               |true               |
|2026-05-17 16:22:09.736|8271261090318600355|7137748156835262556|true               |
|2026-05-17 16:23:45.983|673415979459275383 |8271261090318600355|true               |
|2026-05-17 16:25:09.065|4049912932157741012|673415979459275383 |true               |
|2026-05-17 16:26:34.92 |9080401178567726069|4049912932157741012|true               |
|2026-05-17 16:27:32.462|2138908994108849502|9080401178567726069|true               |
|2026-05-17 16:28:28.279|4952224813326761022|2138908994108849502|true               |
|2026-05-17 16:29:28.045|235206301896660831 |4952224813326761022|true               |
|2026-05-17 16:30:34.858|1423825375660072443|235206301

## 3b. Crecimiento de registros por tag

Cada tag `silver-yellow_taxi-{año}` captura el estado de `silver` tras procesar
todos los meses de ese año. Comparamos también con el tag equivalente en bronze.

In [43]:
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

silver_tags = [
    r["name"] for r in
    refs_df.filter(refs_df[type_col].ilike("tag"))
           .filter(refs_df["name"].rlike(r"^silver-yellow_taxi-\d{4}$"))
           .collect()
]
silver_tags.sort()

# Bronze ingiere ficheros completos del fuente que arrastran filas con
# pickup en años fuera del scope (2001, 2018, ...). Silver las descarta
# por regla, así que para que retention_pct compare manzanas con manzanas
# filtramos bronze al mismo conjunto de años válidos.
VALID_YEARS = [2019, 2020, 2022, 2023, 2025]

tag_pattern = re.compile(r"silver-yellow_taxi-(\d{4})")
rows = []
prev_silver = 0
for tag in silver_tags:
    year = int(tag_pattern.match(tag).group(1))

    nessie.set_spark_branch(spark, tag)
    silver_total = spark.table("nessie.nyc.yellow_taxi").count()

    bronze_tag = f"bronze-yellow_taxi-{year}"
    try:
        nessie.set_spark_branch(spark, bronze_tag)
        bronze_total = (
            spark.table("nessie.nyc.yellow_taxi")
                 .filter(spark_year("tpep_pickup_datetime").isin(VALID_YEARS))
                 .count()
        )
    except Exception:
        bronze_total = None

    retention = round(silver_total / bronze_total * 100, 2) if bronze_total else None
    rows.append((tag, year, bronze_total, silver_total, silver_total - prev_silver, retention))
    prev_silver = silver_total

nessie.set_spark_branch(spark, "silver")

growth_df = spark.createDataFrame(
    rows,
    ["tag", "year", "bronze_records", "silver_records", "records_added", "retention_pct"]
)
growth_df.show(truncate=False)

+-----------------------+----+--------------+--------------+-------------+-------------+
|tag                    |year|bronze_records|silver_records|records_added|retention_pct|
+-----------------------+----+--------------+--------------+-------------+-------------+
|silver-yellow_taxi-2019|2019|22612028      |21926817      |21926817     |96.97        |
|silver-yellow_taxi-2020|2020|38324009      |37125328      |15198511     |96.87        |
|silver-yellow_taxi-2022|2022|47395196      |45665180      |8539852      |96.35        |
|silver-yellow_taxi-2023|2023|56779665      |54573946      |8908766      |96.12        |
|silver-yellow_taxi-2025|2025|67977668      |57423106      |2849160      |84.47        |
+-----------------------+----+--------------+--------------+-------------+-------------+



## 4. Particionado y ficheros

In [44]:
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.partitions").show(truncate=False)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{600}    |0      |6201962     |1         |94180373                     |0                           |0                         |0                           |0                         |2026-05-17 16:25:09.065|4049912932157741012     |
|{601}    |0      |6103376     |1         |92653943         

In [45]:
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df = spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format", "record_count", "file_size_in_bytes"
).show(100, truncate=80)

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|     2849160|          44242128|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2023-0...|    PARQUET|     3226937|          51142686|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2023-0...|    PARQUET|     2764185|          43285032|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2023-0...|    PARQUET|     2917644|          45768784|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tp

## 5. Calidad de datos — visión general

In [46]:
df = spark.table("nessie.nyc.yellow_taxi").cache()
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Total records: 57,423,106
Columns: 20
root
 |-- VendorID: long (nullable = false)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = false)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = false)
 |-- passenger_count: double (nullable = false)
 |-- trip_distance: double (nullable = false)
 |-- RatecodeID: double (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: long (nullable = false)
 |-- DOLocationID: long (nullable = false)
 |-- payment_type: long (nullable = false)
 |-- fare_amount: double (nullable = false)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = false)
 |-- tolls_amount: double (nullable = false)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = false)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [47]:
# Registros por año-mes (valida el particionado y la cobertura temporal)
df.groupBy(
    spark_year("tpep_pickup_datetime").alias("year"),
    spark_month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2019|    1|7460359|
|2019|    2|6833264|
|2019|    3|7633194|
|2020|    1|6201962|
|2020|    2|6103376|
|2020|    3|2893173|
|2022|    1|2326489|
|2022|    2|2798864|
|2022|    3|3414499|
|2023|    1|2917644|
|2023|    2|2764185|
|2023|    3|3226937|
|2025|    1|2849160|
+----+-----+-------+



In [48]:
# Nulos por columna. Los 3 últimos campos (congestion_surcharge, airport_fee,
# cbd_congestion_fee) son evolución de esquema NYC TLC: NULL para años previos
# a su introducción por la fuente, no fallo de la limpieza.
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 0        
 trip_distance         | 0        
 RatecodeID            | 0        
 store_and_fwd_flag    | 0        
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 4707240  
 airport_fee           | 37125328 
 cbd_congestion_fee    | 54573946 



In [49]:
# Estadísticas descriptivas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "tolls_amount", "total_amount", "passenger_count"
).summary("count", "min", "max", "mean", "stddev").show()

+-------+-----------------+------------------+-----------------+-------------------+-----------------+------------------+
|summary|    trip_distance|       fare_amount|       tip_amount|       tolls_amount|     total_amount|   passenger_count|
+-------+-----------------+------------------+-----------------+-------------------+-----------------+------------------+
|  count|         57423106|          57423106|         57423106|           57423106|         57423106|          57423106|
|    min|              0.0|               0.0|              0.0|                0.0|              0.0|               1.0|
|    max|         62359.52|          943274.8|        141492.02|             3288.0|       1084772.17|               6.0|
|   mean|3.024473305397312|14.032787563076907| 2.46661923338341|0.39038326575228544|20.37092040227221|1.5076177523382313|
| stddev|17.90943567829914| 233.2838728865199|18.91794033873773|  1.830318934629187|277.3682734043716|1.0996552888594577|
+-------+---------------

## 6. Validación de las reglas de limpieza

Cada check debe devolver **0 registros**. Si alguno devuelve filas, la regla de limpieza falló.

In [50]:
# Reglas de limpieza consolidadas en una sola pasada sobre df.
# Cada celda devolvía antes una cuenta independiente (1 escaneo por filtro);
# aquí se evalúan todas a la vez y df está cacheado.
VALID_YEARS         = [2019, 2020, 2022, 2023, 2025]
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

checks = {
    "null_pickup":            col("tpep_pickup_datetime").isNull(),
    "null_dropoff":           col("tpep_dropoff_datetime").isNull(),
    "pickup_ge_dropoff":      col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime"),
    "wrong_year":             ~spark_year("tpep_pickup_datetime").isin(VALID_YEARS),
    "neg_trip_distance":      col("trip_distance") < 0,
    "neg_fare_amount":        col("fare_amount")   < 0,
    "neg_total_amount":       col("total_amount")  < 0,
    "neg_tip_amount":         col("tip_amount")    < 0,
    "neg_tolls_amount":       col("tolls_amount")  < 0,
    "invalid_pu_location":    ~col("PULocationID").between(1, 265),
    "invalid_do_location":    ~col("DOLocationID").between(1, 265),
    "invalid_vendor":         ~col("VendorID").isin(VALID_VENDOR_IDS),
    "invalid_ratecode":       ~col("RatecodeID").isin(VALID_RATE_CODES),
    "invalid_payment_type":   ~col("payment_type").isin(VALID_PAYMENT_TYPES),
    "invalid_store_fwd_flag": ~col("store_and_fwd_flag").isin("Y", "N"),
    "invalid_passenger_cnt":  ~col("passenger_count").between(1, 6),
}

results = df.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(label)
    for label, cond in checks.items()
]).collect()[0].asDict()

width = max(len(k) for k in results)
for label, n in results.items():
    flag = "OK" if n == 0 else "FAIL"
    print(f"  [{flag}] {label.ljust(width)} : {n}")

  [OK] null_pickup            : 0
  [OK] null_dropoff           : 0
  [OK] pickup_ge_dropoff      : 0
  [OK] wrong_year             : 0
  [OK] neg_trip_distance      : 0
  [OK] neg_fare_amount        : 0
  [OK] neg_total_amount       : 0
  [OK] neg_tip_amount         : 0
  [OK] neg_tolls_amount       : 0
  [OK] invalid_pu_location    : 0
  [OK] invalid_do_location    : 0
  [OK] invalid_vendor         : 0
  [OK] invalid_ratecode       : 0
  [OK] invalid_payment_type   : 0
  [OK] invalid_store_fwd_flag : 0
  [OK] invalid_passenger_cnt  : 0


In [51]:
# Distribución de los campos categóricos
for field in ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag", "passenger_count"]:
    print(f"--- {field} ---")
    df.groupBy(field).count().orderBy(field).show()

--- VendorID ---


+--------+--------+
|VendorID|   count|
+--------+--------+
|       1|18381557|
|       2|39041549|
+--------+--------+

--- RatecodeID ---


+----------+--------+
|RatecodeID|   count|
+----------+--------+
|       1.0|55236091|
|       2.0| 1529295|
|       3.0|  120714|
|       4.0|   57226|
|       5.0|  359635|
|       6.0|     270|
|      99.0|  119875|
+----------+--------+

--- payment_type ---


+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|43909598|
|           2|13136865|
|           3|  224501|
|           4|  152136|
|           5|       6|
+------------+--------+

--- store_and_fwd_flag ---


+------------------+--------+
|store_and_fwd_flag|   count|
+------------------+--------+
|                 N|56959221|
|                 Y|  463885|
+------------------+--------+

--- passenger_count ---


+---------------+--------+
|passenger_count|   count|
+---------------+--------+
|            1.0|42582255|
|            2.0| 8592285|
|            3.0| 2250059|
|            4.0| 1051657|
|            5.0| 1832636|
|            6.0| 1114214|
+---------------+--------+



## 7. Comparativa Bronze vs Silver por mes

Muestra cuántas filas se eliminaron en la limpieza, mes a mes.

In [52]:
# Solo meses dentro del scope procesado por silver. Bronze trae filas con pickup en
# años/meses fuera de scope (2001, 2018, 2024, 2088, ...) que ensucian la tabla;
# las filtramos para que la comparativa muestre sólo lo que silver intenta procesar.
#
# Cross-branch: la sesión Spark está ligada a una sola ref Nessie a la vez, así que
# materializamos (collect) las cuentas de cada rama antes de cambiar de ref.
#
# clearCache() es imprescindible: la celda 17 cacheó silver.yellow_taxi. El
# CacheManager reutiliza ese InMemoryRelation por igualdad de plan lógico y, como
# las SparkTable de bronze y silver son iguales (mismo nombre Iceberg, snapshotId
# nulo) y la ref Nessie es estado de sesión —no entra en el plan—, spark.table(...)
# sobre bronze recibiría el relation cacheado de silver: la comparativa saldría
# 100% en todos los meses. Limpiar la caché lo evita.
spark.catalog.clearCache()

VALID_YEARS  = [2019, 2020, 2022, 2023, 2025]
VALID_MONTHS = [1, 2, 3]

def monthly_counts(branch: str) -> dict:
    nessie.set_spark_branch(spark, branch)
    rows = (
        spark.table("nessie.nyc.yellow_taxi")
        .filter(spark_year("tpep_pickup_datetime").isin(VALID_YEARS))
        .filter(spark_month("tpep_pickup_datetime").isin(VALID_MONTHS))
        .groupBy(
            spark_year("tpep_pickup_datetime").alias("year"),
            spark_month("tpep_pickup_datetime").alias("month")
        )
        .agg(count("*").alias("n"))
        .collect()
    )
    return {(r["year"], r["month"]): r["n"] for r in rows}

bronze_by_month = monthly_counts("bronze")
silver_by_month = monthly_counts("silver")
nessie.set_spark_branch(spark, "silver")

rows = []
for ym in sorted(bronze_by_month.keys() | silver_by_month.keys()):
    b = bronze_by_month.get(ym, 0)
    s = silver_by_month.get(ym, 0)
    retention = round(s / b * 100, 2) if b else None
    rows.append((ym[0], ym[1], b, s, b - s, retention))

comparison = spark.createDataFrame(
    rows, ["year", "month", "bronze_count", "silver_count", "dropped", "retention_pct"]
).orderBy("year", "month").cache()

comparison.show(50)

+----+-----+------------+------------+-------+-------------+
|year|month|bronze_count|silver_count|dropped|retention_pct|
+----+-----+------------+------------+-------+-------------+
|2019|    1|     7696390|     7460359| 236031|        96.93|
|2019|    2|     7049178|     6833264| 215914|        96.94|
|2019|    3|     7866239|     7633194| 233045|        97.04|
|2020|    1|     6405096|     6201962| 203134|        96.83|
|2020|    2|     6299397|     6103376| 196021|        96.89|
|2020|    3|     3007294|     2893173| 114121|        96.21|
|2022|    1|     2463900|     2326489| 137411|        94.42|
|2022|    2|     2979419|     2798864| 180555|        93.94|
|2022|    3|     3627815|     3414499| 213316|        94.12|
|2023|    1|     3066726|     2917644| 149082|        95.14|
|2023|    2|     2914003|     2764185| 149818|        94.86|
|2023|    3|     3403619|     3226937| 176682|        94.81|
|2025|    1|     3475234|     2849160| 626074|        81.98|
|2025|    2|     3577542

In [53]:
# Totales agregados (reutiliza el comparison cacheado en la celda anterior).
nessie.set_spark_branch(spark, "silver")
comparison.select(
    spark_sum("bronze_count").alias("total_bronze"),
    spark_sum("silver_count").alias("total_silver"),
    spark_sum("dropped").alias("total_dropped"),
    spark_round(
        spark_sum("silver_count") / spark_sum("bronze_count") * 100, 2
    ).alias("overall_retention_pct")
).show()

comparison.unpersist()
_ = df.unpersist()

+------------+------------+-------------+---------------------+
|total_bronze|total_silver|total_dropped|overall_retention_pct|
+------------+------------+-------------+---------------------+
|    67977077|    57423106|     10553971|                84.47|
+------------+------------+-------------+---------------------+



## 7b. Meses en cuarentena

Meses cuyo snapshot WAP falló la auditoría silver y se congeló como tag `review/silver-*`
en lugar de fusionarse. En la comparativa anterior se reconocen porque tienen
`silver_count = 0`, `dropped = bronze_count` y `retention_pct = 0`: bronze los ingirió
pero ninguno llegó a silver.

In [54]:
# Meses en cuarentena: cada tag review/silver-* es un mes que falló la auditoría
# silver. Su snapshot quedó congelado para inspección y NO se fusionó a silver
# -> explica los meses con silver_count = 0 en la comparativa de arriba.
refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

review_tags = sorted(
    r["name"] for r in
    refs_df.filter(refs_df[type_col].ilike("tag"))
           .filter(refs_df["name"].rlike(r"^review/silver-"))
           .collect()
)

if review_tags:
    print(f"Meses en cuarentena ({len(review_tags)}):")
    for t in review_tags:
        print(f"  - {t}")
else:
    print("Sin meses en cuarentena: todos los meses superaron la auditoría silver.")

Meses en cuarentena (2):
  - review/silver-yellow_taxi-2025-02
  - review/silver-yellow_taxi-2025-03


## 7c. Auditoría de cuarentena — regla incumplida

Para cada mes congelado en `review/silver-*` reconstruimos el veredicto de la
auditoría silver (`drop_ratio` frente a `MAX_DROP_RATIO = 20 %`, definido en
`src/pipelines/silver/clean_yellow_taxi.py`) y desglosamos cuántas filas elimina
**cada regla de limpieza por separado**, evaluada de forma aislada sobre el
bronze del mes.

La regla con más filas descartadas es la causa dominante del rechazo. Los
recuentos por regla pueden solaparse — una misma fila puede incumplir varias —,
por lo que su suma supera el total real de filas que elimina `clean()`.

In [55]:
# Reglas y umbral viven en src/pipelines/silver/clean_yellow_taxi.py.
# clearCache() evita la colisión de InMemoryRelation entre refs (ver celda 25).
spark.catalog.clearCache()

MAX_DROP_RATIO      = 0.20
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

# Meses en cuarentena, tomados de la celda 7b (tags review/silver-yellow_taxi-*).
QUARANTINED = [(2025, 2), (2025, 3)]

# Reglas "keep" de clean(): la fila sobrevive sólo si todas son verdaderas.
# when(keep, 0).otherwise(1) cuenta como descartada tanto la fila que incumple
# como la que tiene NULL (la comparación con NULL es falsa -> el filtro la elimina).
RULES = {
    "pickup no nulo":           col("tpep_pickup_datetime").isNotNull(),
    "dropoff no nulo":          col("tpep_dropoff_datetime").isNotNull(),
    "pickup < dropoff":         col("tpep_pickup_datetime") < col("tpep_dropoff_datetime"),
    "trip_distance >= 0":       col("trip_distance") >= 0,
    "fare_amount >= 0":         col("fare_amount") >= 0,
    "total_amount >= 0":        col("total_amount") >= 0,
    "tip_amount >= 0":          col("tip_amount") >= 0,
    "tolls_amount >= 0":        col("tolls_amount") >= 0,
    "PULocationID en [1,265]":  col("PULocationID").between(1, 265),
    "DOLocationID en [1,265]":  col("DOLocationID").between(1, 265),
    "VendorID válido":          col("VendorID").isin(VALID_VENDOR_IDS),
    "RatecodeID válido":        col("RatecodeID").isin(VALID_RATE_CODES),
    "payment_type válido":      col("payment_type").isin(VALID_PAYMENT_TYPES),
    "store_and_fwd_flag (Y/N)": col("store_and_fwd_flag").isin("Y", "N"),
    "passenger_count en [1,6]": col("passenger_count").between(1, 6),
}

for yr, mo in QUARANTINED:
    tag = f"review/silver-yellow_taxi-{yr}-{mo:02d}"
    month = (spark_year("tpep_pickup_datetime") == yr) & (spark_month("tpep_pickup_datetime") == mo)

    # Bronze del mes: mismo filtro year+month que aplica el pipeline.
    nessie.set_spark_branch(spark, "bronze")
    bm = spark.table("nessie.nyc.yellow_taxi").filter(month).cache()
    bronze_count = bm.count()

    # Filas descartadas por cada regla, evaluada de forma aislada sobre bronze.
    per_rule = bm.agg(*[
        spark_sum(when(keep, 0).otherwise(1)).alias(name)
        for name, keep in RULES.items()
    ]).collect()[0].asDict()
    # dropDuplicates() no es una condición de columna: se mide aparte.
    per_rule["filas duplicadas"] = bronze_count - bm.dropDuplicates().count()
    bm.unpersist()

    # Veredicto real: el tag review/ congela el snapshot WAP ya limpiado, así que
    # su recuento del mes es el silver_count que evaluó validate().
    nessie.set_spark_branch(spark, tag)
    silver_count = spark.table("nessie.nyc.yellow_taxi").filter(month).count()

    drop_ratio = (bronze_count - silver_count) / bronze_count
    dominant = max(per_rule, key=per_rule.get)

    print(f"\n=== {yr}-{mo:02d}  ({tag}) ===")
    print(f"  Bronze: {bronze_count:,}   Silver tras clean(): {silver_count:,}")
    print(f"  Drop ratio: {drop_ratio:.1%}   Umbral: {MAX_DROP_RATIO:.0%}   "
          f"-> {'RECHAZADO' if drop_ratio > MAX_DROP_RATIO else 'OK'}")
    print("  Filas descartadas por regla (aislada; los solapamientos hacen que\n"
          "  la suma supere el total real de clean()):")
    for name, n in sorted(per_rule.items(), key=lambda kv: -kv[1]):
        marca = "  <-- regla dominante" if name == dominant else ""
        print(f"    {name:<26} {n:>10,}  ({n / bronze_count * 100:5.2f} %){marca}")

_ = nessie.set_spark_branch(spark, "silver")


=== 2025-02  (review/silver-yellow_taxi-2025-02) ===
  Bronze: 3,577,542   Silver tras clean(): 2,688,853
  Drop ratio: 24.8%   Umbral: 20%   -> RECHAZADO
  Filas descartadas por regla (aislada; los solapamientos hacen que
  la suma supere el total real de clean()):
    passenger_count en [1,6]      828,704  (23.16 %)  <-- regla dominante
    RatecodeID válido             806,937  (22.56 %)
    store_and_fwd_flag (Y/N)      806,937  (22.56 %)
    fare_amount >= 0              182,655  ( 5.11 %)
    total_amount >= 0              55,179  ( 1.54 %)
    pickup < dropoff                5,164  ( 0.14 %)
    tolls_amount >= 0               4,138  ( 0.12 %)
    tip_amount >= 0                   105  ( 0.00 %)
    pickup no nulo                      0  ( 0.00 %)
    dropoff no nulo                     0  ( 0.00 %)
    trip_distance >= 0                  0  ( 0.00 %)
    PULocationID en [1,265]             0  ( 0.00 %)
    DOLocationID en [1,265]             0  ( 0.00 %)
    VendorID válido   


=== 2025-03  (review/silver-yellow_taxi-2025-03) ===
  Bronze: 4,145,225   Silver tras clean(): 3,115,214
  Drop ratio: 24.8%   Umbral: 20%   -> RECHAZADO
  Filas descartadas por regla (aislada; los solapamientos hacen que
  la suma supere el total real de clean()):
    passenger_count en [1,6]      939,391  (22.66 %)  <-- regla dominante
    RatecodeID válido             916,663  (22.11 %)
    store_and_fwd_flag (Y/N)      916,663  (22.11 %)
    fare_amount >= 0              208,724  ( 5.04 %)
    total_amount >= 0              68,686  ( 1.66 %)
    pickup < dropoff               22,280  ( 0.54 %)
    tolls_amount >= 0               5,697  ( 0.14 %)
    tip_amount >= 0                   131  ( 0.00 %)
    pickup no nulo                      0  ( 0.00 %)
    dropoff no nulo                     0  ( 0.00 %)
    trip_distance >= 0                  0  ( 0.00 %)
    PULocationID en [1,265]             0  ( 0.00 %)
    DOLocationID en [1,265]             0  ( 0.00 %)
    VendorID válido   

### Conclusión

Ambos meses se rechazan con un **drop ratio del 24,8 %**, por encima del umbral del
20 %. La regla dominante es **`passenger_count` en [1, 6]** — 23,2 % de las filas en
febrero, 22,7 % en marzo —, que por sí sola ya supera el umbral.

El dato clave es que `RatecodeID válido` y `store_and_fwd_flag (Y/N)` descartan
**exactamente el mismo número de filas** (806 937 en febrero, 916 663 en marzo), y el
total descartado (24,8 %) apenas supera a la regla dominante. Las tres reglas grandes
golpean prácticamente las mismas filas: un único bloque entr 830 mil y 940 mil viajes de
2025 (aproximadamente el 23–24 % del mes) en los que `passenger_count`, `RatecodeID` y
`store_and_fwd_flag` faltan a la vez.

No son tres problemas de calidad independientes, sino un **cambio en el formato de
reporte de 2025**: una clase de viajes que no informa esos tres campos derivados del
taxímetro (la auditoría cuenta los NULL como descartes). La misma causa, con menor
magnitud, dejó a enero de 2025 al borde del umbral — 18 % descartado, aprobado por poco.

`fare_amount >= 0` (~5 %) es un problema aparte y menor: tarifas negativas presentes en
el bronze de 2025.

## 8. Muestra de datos

In [56]:
spark.table("nessie.nyc.yellow_taxi").show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-01-01 00:32:40 |2025-01-01 00:35:13  |1.0            |0.5          |1.0       |N                 |236         |237 

## 9. Taxi zone lookup — esquema y calidad


In [57]:
zones = spark.table("nessie.nyc.taxi_zone_lookup")
print(f"Records: {zones.count()}")
zones.printSchema()
spark.sql("DESCRIBE EXTENDED nessie.nyc.taxi_zone_lookup").show(50, truncate=False)


Records: 265
root
 |-- location_id: integer (nullable = false)
 |-- borough: string (nullable = false)
 |-- zone: string (nullable = false)
 |-- service_zone: string (nullable = true)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                                                     |comment|
+-----------------------------+--------------------------------------------------------------------------------------------------

In [58]:
# Nulos en campos críticos y rango del location_id
print("Null location_id:", zones.filter(col("location_id").isNull()).count())
print("Null borough   :", zones.filter(col("borough").isNull() | (col("borough") == "")).count())
print("Null zone      :", zones.filter(col("zone").isNull() | (col("zone") == "")).count())
print("Out-of-range location_id:",
      zones.filter(~col("location_id").between(1, 265)).count())

# Duplicados por location_id (clave lógica)
dupes = (zones.groupBy("location_id").count()
         .filter(col("count") > 1))
n_dupes = dupes.count()
print("Duplicate location_id:", n_dupes)
if n_dupes > 0:
    dupes.show()

Null location_id: 0
Null borough   : 0
Null zone      : 0
Out-of-range location_id: 0
Duplicate location_id: 0


In [59]:
# Distribución por borough y service_zone
print("--- borough ---")
zones.groupBy("borough").count().orderBy("borough").show(truncate=False)
print("--- service_zone ---")
zones.groupBy("service_zone").count().orderBy("service_zone").show(truncate=False)


--- borough ---
+-------------+-----+
|borough      |count|
+-------------+-----+
|Bronx        |43   |
|Brooklyn     |61   |
|EWR          |1    |
|Manhattan    |69   |
|N/A          |1    |
|Queens       |69   |
|Staten Island|20   |
|Unknown      |1    |
+-------------+-----+

--- service_zone ---
+------------+-----+
|service_zone|count|
+------------+-----+
|Airports    |2    |
|Boro Zone   |205  |
|EWR         |1    |
|N/A         |2    |
|Yellow Zone |55   |
+------------+-----+



In [60]:
# Comparativa Bronze vs Silver (zone lookup es pequeno, ~265 filas: comparamos conjuntos de IDs)
nessie.set_spark_branch(spark, "bronze")
bronze_ids = {row["LocationID"] for row in
              spark.table("nessie.nyc.taxi_zone_lookup").select("LocationID").collect()}
nessie.set_spark_branch(spark, "silver")
silver_ids = {row["location_id"] for row in zones.select("location_id").collect()}

b, s = len(bronze_ids), len(silver_ids)
if b:
    print(f"Bronze: {b}  Silver: {s}  Dropped: {b - s}  Retention: {s / b:.1%}")
else:
    print("Bronze empty")

missing = sorted(bronze_ids - silver_ids)
print(f"IDs in bronze but not in silver: {len(missing)}")
if missing:
    print("  " + ", ".join(str(i) for i in missing))


Bronze: 265  Silver: 265  Dropped: 0  Retention: 100.0%
IDs in bronze but not in silver: 0


In [61]:
# Integridad referencial: ¿todas las PULocationID/DOLocationID de silver.yellow_taxi
# existen en silver.taxi_zone_lookup?
trips = spark.table("nessie.nyc.yellow_taxi")

pu_missing = (trips.select(col("PULocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))
do_missing = (trips.select(col("DOLocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))

print("PULocationID not found in zone lookup:", pu_missing.count())
pu_missing.show(truncate=False)
print("DOLocationID not found in zone lookup:", do_missing.count())
do_missing.show(truncate=False)


PULocationID not found in zone lookup: 0


+-----------+
|location_id|
+-----------+
+-----------+



DOLocationID not found in zone lookup: 0


+-----------+
|location_id|
+-----------+
+-----------+



In [62]:
# Muestra
zones.orderBy("location_id").show(10, truncate=False)


+-----------+-------------+-----------------------+------------+
|location_id|borough      |zone                   |service_zone|
+-----------+-------------+-----------------------+------------+
|1          |EWR          |Newark Airport         |EWR         |
|2          |Queens       |Jamaica Bay            |Boro Zone   |
|3          |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4          |Manhattan    |Alphabet City          |Yellow Zone |
|5          |Staten Island|Arden Heights          |Boro Zone   |
|6          |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7          |Queens       |Astoria                |Boro Zone   |
|8          |Queens       |Astoria Park           |Boro Zone   |
|9          |Queens       |Auburndale             |Boro Zone   |
|10         |Queens       |Baisley Park           |Boro Zone   |
+-----------+-------------+-----------------------+------------+
only showing top 10 rows

